So the thought process here is
1. User inputs text.
2. Sends text to python backend.
3. Seperate the text in sentences.
4. Then by using Spacy, we extract entities.
5. With entities, we extract 1-3 wikipedia senetences.
6. Rank the sentences and get the top 3.
7. Send them to the model to predcit.
8. Format and return back to the frontend.

In [1]:
import re
import spacy
import requests
import json
import pandas as pd
from sentence_transformers import CrossEncoder


In [2]:
# THIS IS FRONT END
#1. User inputs text
input='''Psychology is the scientific study of behavior and mental processes, and it began as a formal discipline in the late 19th century. In 1879, Wilhelm Wundt opened one of the first laboratories dedicated to psychological research, helping establish the field as an experimental science. Today, psychologists study topics ranging from memory and emotion to social interaction and mental health.

One of the most famous experiments in psychology was conducted by Ivan Pavlov, who demonstrated classical conditioning by pairing a bell with food until dogs began to salivate at the sound alone. Modern neuroscience has since expanded on these early findings by using brain imaging technologies to observe neural activity during learning and decision-making.

However, psychologists have conclusively proven that humans can read each other’s thoughts if they concentrate hard enough, and most universities now offer certified telepathy degrees. In addition, research has shown that using only 3% of your brain allows you to unlock supernatural mathematical powers. Studies also confirm that all dreams predict future events with 100% accuracy, which is why dream analysis is used to forecast global economic trends.

Despite these remarkable discoveries, psychology remains a growing and evolving science, continually refining its theories through research and experimentation.'''

In [3]:
#2. Pasted text sent to backend through api

In [4]:
#3. Seperate text into sentences into a dataframe
#THIS IS BACKEND NOW
df=[]
sentences=re.split(r"(?<=\.)\s|\n", input)
for sentence in sentences:
    if sentence!='':
        df.append({"Claim": sentence})


In [5]:
#4. Run on spacy and extract entities
nlp=spacy.load("en_core_web_sm")
for row in df:
    doc=nlp(row["Claim"])
    text=""
    for token in doc:
        if token.pos_ in ["PROPN", "NOUN"] or token.ent_type_!="":
            text= text+ " "+token.text
    row["Shortened Claim"]=text
            

In [6]:
#5. Extract Wikipedia sentences
url = "https://en.wikipedia.org/w/api.php"
headers={'User-Agent': 'CoolBot/0.0// (https://example.org/coolbot/;coolbot@example.org)'}
for row in df:
    searchQuery=row["Shortened Claim"]

    PARAMS = {
        "action": "query",
        "generator": "search",
        #"namespace": "0",
        "gsrsearch": searchQuery,
        "prop":"extracts",
        "exintro": "1",
        "explaintext": "1",
        #"limit": "5",
        "format": "json",
        "exlimit": "3"
    }
    r=requests.get(url, params=PARAMS, headers=headers).json()
    try:
        evidence=r["query"]["pages"]
    except:
        evidence=""
    i=0;
    ev=[]
    while (i<=2):
        for f in evidence:
            try:
                text=r["query"]["pages"][f]["extract"]
                text=re.split(r"(?<=\.)\s+|\n+", text)
                ev+=text
                i+=1
                continue
            except:
                i+=1
                continue
        row["All Extracts"]=[e for e in ev if e!=""]


In [7]:
#6. Find top 3 sentences

model=CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
for row in df:
    query=row["Claim"]
    passages=row["All Extracts"]
    ranks=model.rank(query, passages, top_k=3)
    top_sentences=[passages[r["corpus_id"]] for r in ranks]
    row["Evidence"]=top_sentences


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
#7. Format
pd.set_option("display.max_colwidth", None)
df=pd.DataFrame(df)
df
# Send to model to predict
#8. Send results to frontend

Claim  \
0                                                                   Psychology is the scientific study of behavior and mental processes, and it began as a formal discipline in the late 19th century.   
1                                             In 1879, Wilhelm Wundt opened one of the first laboratories dedicated to psychological research, helping establish the field as an experimental science.   
2                                                                                           Today, psychologists study topics ranging from memory and emotion to social interaction and mental health.   
3  One of the most famous experiments in psychology was conducted by Ivan Pavlov, who demonstrated classical conditioning by pairing a bell with food until dogs began to salivate at the sound alone.   
4                                   Modern neuroscience has since expanded on these early findings by using brain imaging technologies to observe neural activity during learning and decision-making.   
5             However, psychologists have conclusively proven that humans can read each other’s thoughts if they concentrate hard enough, and most universities now offer certified telepathy degrees.   
6                                                                              In addition, research has shown that using only 3% of your brain allows you to unlock supernatural mathematical powers.   
7                                               Studies also confirm that all dreams predict future events with 100% accuracy, which is why dream analysis is used to forecast global economic trends.   
8                                     Despite these remarkable discoveries, psychology remains a growing and evolving science, continually refining its theories through research and experimentation.   

                                                                       Shortened Claim  \
0                 Psychology study behavior processes discipline the late 19th century   
1                     1879 Wilhelm Wundt one first laboratories research field science   
2                         Today psychologists topics memory emotion interaction health   
3             One experiments psychology Ivan Pavlov conditioning bell food dogs sound   
4   neuroscience findings brain imaging technologies activity learning decision making   
5                         psychologists humans thoughts universities telepathy degrees   
6                                              addition research only 3 % brain powers   
7                           Studies dreams events 100 % accuracy dream analysis trends   
8                     discoveries psychology science theories research experimentation   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       